# Chronic Disease Burden and Medicare Utilization

**Population:** Medicare fee-for-service beneficiaries. **Source:** CMS chronic-conditions aggregation supplied with BMI 5007. This is a descriptive analysis, not population-wide disease prevalence or causal attribution.

In [7]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src" / "analyze.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from analyze import load_data, prevalence_comparison, utilization_comparison, extrema, subgroup_prevalence
df = load_data()
print(f"Rows: {len(df):,} | Chronic conditions: {df.Bene_Cond.nunique()}")
df.head()

Rows: 32,340 | Chronic conditions: 21


,Bene_Geo_Lvl,Bene_Geo_Desc,Bene_Geo_Cd,Bene_Age_Lvl,Bene_Demo_Lvl,Bene_Demo_Desc,Bene_Cond,Prvlnc,Tot_Mdcr_Stdzd_Pymt_PC,Tot_Mdcr_Pymt_PC,Hosp_Readmsn_Rate,ER_Visits_Per_1000_Benes
0,State,Alabama,01,All,All,All,Alcohol Abuse,0.0188,25102.3405,23348.6039,0.2413,2184.7557
1,State,Alabama,01,65+,Dual Status,Medicare Only,Alcohol Abuse,0.0118,NaN,NaN,NaN,NaN
2,State,Alabama,01,<65,Dual Status,Medicare Only,Alcohol Abuse,0.0320,NaN,NaN,NaN,NaN
3,State,Alabama,01,All,Dual Status,Medicare Only,Alcohol Abuse,0.0147,NaN,NaN,NaN,NaN
4,State,Alabama,01,65+,Dual Status,Medicare and Medicaid,Alcohol Abuse,0.0238,NaN,NaN,NaN,NaN


## 1. Chronic disease prevalence

`Prvlnc` measures the share of Medicare FFS beneficiaries in the respective condition/demographic aggregate. We convert this share to percentages. Compare **like-for-like overall strata**, not overlapping subgroup rows.

In [8]:
prev = prevalence_comparison(df)
conditions = ["Diabetes", "Hypertension", "COPD", "Heart Failure", "Depression"]
prev_table = prev[prev.Bene_Cond.isin(conditions)].pivot(index="Bene_Cond", columns="Bene_Geo_Desc", values="Prevalence_pct").reindex(conditions)
display(prev_table.round(2))
ax = prev_table.plot.bar(figsize=(11, 5.5))
ax.set(xlabel="Chronic condition", ylabel="Prevalence (%)", title="Prevalence among Medicare FFS beneficiaries")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()

Bene_Geo_Desc,California,Florida,National,New York,Texas
Bene_Cond,,,,,
Diabetes,27.19,27.81,26.95,31.34,28.79
Hypertension,52.96,62.38,57.20,57.67,59.87
COPD,9.52,13.46,11.47,11.05,11.15
Heart Failure,13.91,14.78,13.95,15.01,15.60
Depression,16.22,19.48,18.40,17.02,18.19


C:\temp\ipykernel_29268\3352533282.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### Age and race distributions

Race-level and age-level aggregates are separate slices in the dataset. They should not be interpreted as a joint age-by-race distribution.

In [9]:
age = df[(df.Bene_Geo_Lvl=="National") & (df.Bene_Demo_Lvl=="All") & df.Bene_Age_Lvl.isin(["<65", "65+"])].copy()
age["Prevalence_pct"] = age.Prvlnc * 100
display(age[age.Bene_Cond.isin(conditions)][["Bene_Cond", "Bene_Age_Lvl", "Prevalence_pct"]].pivot(index="Bene_Cond", columns="Bene_Age_Lvl", values="Prevalence_pct").round(2))
race = subgroup_prevalence(df, "Race")
display(race[race.Bene_Cond=="Diabetes"].round(2))

Bene_Age_Lvl
Bene_Cond


,Bene_Cond,Bene_Demo_Desc,Prevalence_pct
23672,Diabetes,Asian Pacific Islander,36.51
23675,Diabetes,Hispanic,37.56
23678,Diabetes,Native American,40.11
23681,Diabetes,non-Hispanic Black,37.63
23684,Diabetes,non-Hispanic White,24.64


## 2. Medicare utilization

Spending, readmission rates, and emergency department utilization are not attributable exclusively to the named condition. We focus on the overall age/demographic strata where these measurements are generally reported.

In [10]:
util = utilization_comparison(df)
cols = ["Tot_Mdcr_Stdzd_Pymt_PC", "Tot_Mdcr_Pymt_PC", "Hosp_Readmsn_Rate", "ER_Visits_Per_1000_Benes"]
display(util[(util.Bene_Geo_Desc=="National") & util.Bene_Cond.isin(conditions)][["Bene_Cond", *cols]].round(2))
spending = util[(util.Bene_Geo_Desc=="National") & util.Bene_Cond.isin(conditions)].set_index("Bene_Cond")["Tot_Mdcr_Stdzd_Pymt_PC"].reindex(conditions)
ax = spending.plot.bar(figsize=(9, 5))
ax.set(xlabel="Condition", ylabel="Standardized payments per beneficiary (USD)", title="National Medicare spending by condition cohort")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()

,Bene_Cond,Tot_Mdcr_Stdzd_Pymt_PC,Tot_Mdcr_Pymt_PC,Hosp_Readmsn_Rate,ER_Visits_Per_1000_Benes
7449,COPD,25446.52,27255.40,0.24,1736.66
7488,Depression,20471.68,21810.74,0.22,1462.40
7501,Diabetes,16272.44,17380.29,0.22,1026.77
7540,Heart Failure,28604.45,30940.20,0.24,1731.18
7579,Hypertension,14580.63,15514.08,0.19,948.20


C:\temp\ipykernel_29268\2456247748.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 3. State-wise prevalence relative to national benchmarks

Compare the highest and lowest **available state estimates** with the corresponding national aggregate. A difference in prevalence is expressed in **percentage points**. Rankings do not adjust for demographic differences or missing/suppressed strata.

In [11]:
ext = extrema(df)
display(ext[["Bene_Cond", "Highest_State", "Highest_Pct", "Lowest_State", "Lowest_Pct", "National_Pct"]].round(2).head(15))
ranked = ext.nlargest(10, "Highest_vs_National_pp").sort_values("Highest_vs_National_pp")
ax = ranked.plot.barh(x="Bene_Cond", y="Highest_vs_National_pp", legend=False, figsize=(10, 6))
ax.set(xlabel="Maximum state minus national (percentage points)", ylabel="Condition", title="Largest state deviations from national prevalence")
plt.tight_layout(); plt.show()

,Bene_Cond,Highest_State,Highest_Pct,Lowest_State,Lowest_Pct,National_Pct
0,Alcohol Abuse,Massachusetts,3.69,Utah,1.40,2.08
1,Alzheimer's Disease/Dementia,Connecticut,12.89,Wyoming,7.70,10.78
2,Arthritis,Alabama,38.97,Hawaii,21.29,33.47
3,Asthma,Rhode Island,6.66,Wyoming,3.08,4.97
4,Atrial Fibrillation,Connecticut,10.72,New Mexico,5.39,8.41
5,Autism Spectrum Disorders,Minnesota,0.76,Arizona,0.10,0.26
6,COPD,West Virginia,17.55,Hawaii,6.58,11.47
7,Cancer,Florida,10.14,New Mexico,6.19,8.36
8,Chronic Kidney Disease,Florida,28.21,Wyoming,15.62,24.48
9,Depression,Massachusetts,23.18,Hawaii,8.99,18.40


C:\temp\ipykernel_29268\2873225336.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### Demographic comparisons

Sex and race are analyzed separately because the source reports separate aggregates. Missing/suppressed state groups are excluded from the extrema calculation.

In [12]:
sex = extrema(df, "Sex")
race_ext = extrema(df, "Race")
print("Sex-stratified results:")
display(sex.head(8).round(2))
print("Race-stratified results:")
display(race_ext.head(8).round(2))

Sex-stratified results:


,Bene_Cond,Bene_Demo_Desc,Highest_State,Highest_Prev,Lowest_State,Lowest_Prev,National_Prev,Highest_Pct,Lowest_Pct,National_Pct,Highest_vs_National_pp,Lowest_vs_National_pp
0,Alcohol Abuse,Female,Minnesota,0.02,Hawaii,0.01,0.01,2.38,0.76,1.19,1.19,-0.43
1,Alcohol Abuse,Male,Massachusetts,0.05,Utah,0.02,0.03,5.48,2.03,3.15,2.33,-1.12
2,Alzheimer's Disease/Dementia,Female,Connecticut,0.14,Arizona,0.09,0.12,14.38,8.63,12.21,2.17,-3.58
3,Alzheimer's Disease/Dementia,Male,Connecticut,0.11,Wyoming,0.06,0.09,10.99,6.25,9.06,1.93,-2.81
4,Arthritis,Female,Alabama,0.45,Hawaii,0.25,0.39,45.10,24.94,38.72,6.38,-13.78
5,Arthritis,Male,Alabama,0.32,Hawaii,0.17,0.27,31.53,17.26,27.14,4.39,-9.88
6,Asthma,Female,Rhode Island,0.09,Wyoming,0.04,0.06,8.52,4.06,6.39,2.13,-2.33
7,Asthma,Male,Massachusetts,0.05,Wyoming,0.02,0.03,4.60,2.01,3.26,1.34,-1.25


Race-stratified results:


,Bene_Cond,Bene_Demo_Desc,Highest_State,Highest_Prev,Lowest_State,Lowest_Prev,National_Prev,Highest_Pct,Lowest_Pct,National_Pct,Highest_vs_National_pp,Lowest_vs_National_pp
0,Alcohol Abuse,Asian Pacific Islander,Maine,0.03,Maryland,0.01,0.01,2.55,0.54,0.79,1.76,-0.25
1,Alcohol Abuse,Hispanic,Maine,0.05,South Carolina,0.01,0.02,4.77,1.31,2.18,2.59,-0.87
2,Alcohol Abuse,Native American,New Hampshire,0.11,Arkansas,0.02,0.05,11.02,2.40,4.88,6.14,-2.48
3,Alcohol Abuse,non-Hispanic Black,Minnesota,0.09,Arizona,0.02,0.03,8.61,2.17,2.96,5.65,-0.79
4,Alcohol Abuse,non-Hispanic White,Massachusetts,0.04,Utah,0.01,0.02,3.63,1.33,2.01,1.62,-0.68
5,Alzheimer's Disease/Dementia,Asian Pacific Islander,New York,0.14,Minnesota,0.05,0.11,14.15,4.73,10.53,3.62,-5.80
6,Alzheimer's Disease/Dementia,Hispanic,Florida,0.19,Vermont,0.06,0.11,19.11,5.90,10.96,8.15,-5.06
7,Alzheimer's Disease/Dementia,Native American,New Jersey,0.13,Kentucky,0.07,0.10,13.15,6.96,9.77,3.38,-2.81


## Interpretation and limitations

These descriptive results can identify conditions and locations warranting further investigation. They **cannot** establish causes of chronic disease, estimate prevalence in all US residents, or determine appropriate staffing levels without local case-mix and capacity information. CMS payment/use measures encompass care for comorbid conditions. The dataset release year and any disclosure suppression rules must be confirmed from official CMS metadata before external publication.